In [3]:
# TASK 1 - Data Processing

"""
MasakhaNEWS — Task 1: Data processing.
Loads eng/xho/sna splits, cleans + tokenises headline+text, and builds
TF-IDF feature matrices for a multinomial logistic regression model.
"""

import os
import re
import json
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

DATA_DIR = "data"
OUT_DIR = "features"
LANGS = ["eng", "xho", "sna"]
SPLITS = ["train", "dev", "test"]

URL_RE = re.compile(r"https?://\S+|www\.\S+")
DIGIT_RE = re.compile(r"\d+")
TOKEN_RE = re.compile(r"<num>|[a-z]+")  # text is already lowercased by clean_text()


def load_split(lang: str, split: str) -> pd.DataFrame:
    """Read one tsv file and build the full_text = headline + text column."""
    df = pd.read_csv(os.path.join(DATA_DIR, lang, f"{split}.tsv"), sep="\t")
    df["full_text"] = df["headline"].fillna("") + " " + df["text"].fillna("")
    return df[["category", "full_text"]]


def load_language(lang: str) -> dict:
    """Load train/dev/test for one language into a dict of DataFrames."""
    return {split: load_split(lang, split) for split in SPLITS}


def clean_text(text: str) -> str:
    """Lowercase, drop URLs, collapse digit runs to a shared <num> token."""
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = DIGIT_RE.sub(" <num> ", text)
    return text


def tokenize(text: str) -> list:
    """Split into letter-only tokens (plus <num> placeholders), applied
    identically across all 3 languages so no language gets special-cased
    preprocessing (xho/sna have no supported stemmer/stopword list anyway)."""
    return TOKEN_RE.findall(clean_text(text))


def build_vectorizer() -> TfidfVectorizer:
    """Construct an (unfitted) TF-IDF vectorizer: unigrams+bigrams, min_df=2
    to drop one-off noise tokens, sublinear (log-scaled) term frequency."""
    return TfidfVectorizer(
        tokenizer=tokenize,
        lowercase=False,
        token_pattern=None,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )


def process_language(lang: str, stats: dict):
    """Fit/transform features + labels for one language, save them to disk,
    and record summary stats (sizes, classes, vocab size) into `stats`."""
    print(f"\n=== {lang} ===")
    data = load_language(lang)

    vectorizer = build_vectorizer()
    X_train = vectorizer.fit_transform(data["train"]["full_text"])  # fit on train only
    X_dev = vectorizer.transform(data["dev"]["full_text"])
    X_test = vectorizer.transform(data["test"]["full_text"])

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(data["train"]["category"])
    y_dev = label_encoder.transform(data["dev"]["category"])
    y_test = label_encoder.transform(data["test"]["category"])

    out_dir = os.path.join(OUT_DIR, lang)
    os.makedirs(out_dir, exist_ok=True)
    sparse.save_npz(os.path.join(out_dir, "X_train.npz"), X_train)
    sparse.save_npz(os.path.join(out_dir, "X_dev.npz"), X_dev)
    sparse.save_npz(os.path.join(out_dir, "X_test.npz"), X_test)
    np.save(os.path.join(out_dir, "y_train.npy"), y_train)
    np.save(os.path.join(out_dir, "y_dev.npy"), y_dev)
    np.save(os.path.join(out_dir, "y_test.npy"), y_test)
    joblib.dump(vectorizer, os.path.join(out_dir, "vectorizer.joblib"))
    joblib.dump(label_encoder, os.path.join(out_dir, "label_encoder.joblib"))

    lang_stats = {
        "n_train": X_train.shape[0],
        "n_dev": X_dev.shape[0],
        "n_test": X_test.shape[0],
        "n_classes": len(label_encoder.classes_),
        "classes": list(label_encoder.classes_),
        "vocab_size": len(vectorizer.vocabulary_),
        "train_class_counts": data["train"]["category"].value_counts().to_dict(),
    }
    stats[lang] = lang_stats
    print(f"  sizes: {lang_stats['n_train']}/{lang_stats['n_dev']}/{lang_stats['n_test']}")
    print(f"  classes ({lang_stats['n_classes']}): {lang_stats['classes']}")
    print(f"  vocab size: {lang_stats['vocab_size']}")
    print(f"  train class distribution: {lang_stats['train_class_counts']}")


def main():
    """Run the pipeline for all languages and write a summary stats file."""
    os.makedirs(OUT_DIR, exist_ok=True)
    stats = {}
    for lang in LANGS:
        process_language(lang, stats)
    with open(os.path.join(OUT_DIR, "summary_stats.json"), "w") as f:
        json.dump(stats, f, indent=2, default=str)
    print(f"\nSaved feature matrices + stats to ./{OUT_DIR}/")


if __name__ == "__main__":
    main()


=== eng ===
  sizes: 3309/472/948
  classes (6): ['business', 'entertainment', 'health', 'politics', 'sports', 'technology']
  vocab size: 176037
  train class distribution: {'sports': 700, 'politics': 574, 'business': 559, 'entertainment': 525, 'health': 522, 'technology': 429}

=== xho ===
  sizes: 1032/147/297
  classes (5): ['business', 'entertainment', 'health', 'politics', 'sports']
  vocab size: 34012
  train class distribution: {'entertainment': 350, 'sports': 347, 'politics': 215, 'health': 70, 'business': 50}

=== sna ===
  sizes: 1288/185/369
  classes (4): ['business', 'health', 'politics', 'sports']
  vocab size: 44893
  train class distribution: {'business': 350, 'politics': 350, 'health': 297, 'sports': 291}

Saved feature matrices + stats to ./features/
